In [1]:
import pandas as pd
import random

In [2]:

import os


# Change to working folder for this week:
base_path = '/Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/week_3'

os.chdir(base_path)

print(os.getcwd())

/Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/week_3


# Week 3 Lab: Building NLP Data Pipelines with Hugging Face Datasets


This notebook we will look at the Hugging Face `datasets` library.

By the end, we will go from raw text files to a model-ready dataset that works with open source models on the Hugging Face Hub.


In particular by then end of this notebook we should be able to:

- Load dataset files and inspect schema
- Create train/validation/test splits safely
- Clean text with `map` and `filter`
- Tokenize text for transformer models
- Save and share datasets in a reusable format


## Why use Hugging Face Datasets?



If you skip a data framework and do things in an adhoc manner your project often becomes fragile:
- columns silently change,
- train/test splits are inconsistent,
- preprocessing cannot be reproduced,
- model training code needs custom glue.

`datasets` solves this with a standard format and a consistent API.



### Practical benefits


- One API for many data sources: Hub datasets, local CSV/JSON/TXT, in-memory Python lists.
- Typed schema: `features` helps you catch data issues early.
- Model compatibility: easy path to columns expected by Hugging Face training APIs.
- Reproducibility: save to disk and reload exact processed data.

## Why put your own data in `Dataset` format?

Even if your data starts in CSV or a database, converting to `Dataset` is useful because:

- You get **repeatable preprocessing** (same operations every run).
- You can **share exact versions** with teammates.
- You can **switch models** without rewriting your data pipeline.
- You can **publish to Hub** so training notebooks can load your data with one line.

This is one of the biggest productivity gains when working with open source models.


## 1. Setup


In this notebook we'll use the Hugging Faces **Datasets** library to handle data in a way that is fast, consistent, and reproducible.

In [3]:
from datasets import Dataset, DatasetDict, load_dataset

/Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Why these imports?

We import three core pieces above:

- `Dataset`: a single dataset object (basically a table of data (like a DataFrame) with named columns (“features”) and rows (“examples”)) that supports operations like `map`, `filter`, `shuffle`, and `train_test_split`.
- `DatasetDict`: a container holding multiple datasets, typically by split, e.g. `{"train": ..., "validation": ..., "test": ...}`. This is what you typically get back from `load_dataset()` when a dataset has predefined splits.
- `load_dataset`: the main helper that loads datasets from the Hugging Face Hub, built-in dataset scripts, or local files — often returning a `DatasetDict` with standard splits.

These will are important when we start focusing on data sourcing, and train/validation/test splits.


The `Dataset` object is based on Apache Arrow, which defines a typed columnar format that is more memory efficient than native Python. In particular, we are able to work with large datasets on machines with small amounts of memory. The section below taken from the Hugging Face Datasets docs gives you more detail on Arrow.

### Datasets 🤝 Arrow - Optional Reading


#### What is Arrow?

[Arrow](https://arrow.apache.org/) enables large amounts of data to be processed and moved quickly. It is a specific data format that stores data in a columnar memory layout. This provides several significant advantages:

* Arrow's standard format allows [zero-copy reads](https://en.wikipedia.org/wiki/Zero-copy) which removes virtually all serialization overhead.
* Arrow is language-agnostic so it supports different programming languages.
* Arrow is column-oriented so it is faster at querying and processing slices or columns of data.
* Arrow allows for copy-free hand-offs to standard machine learning tools such as NumPy, Pandas, PyTorch, and TensorFlow.
* Arrow supports many, possibly nested, column types.

#### Memory-mapping

🤗 Datasets uses Arrow for its local caching system. It allows datasets to be backed by an on-disk cache, which is memory-mapped for fast lookup.
This architecture allows for large datasets to be used on machines with relatively small device memory.

For example, loading the full English Wikipedia dataset only takes a few MB of RAM:

```python
>>> import os; import psutil; import timeit
>>> from datasets import load_dataset

# Process.memory_info is expressed in bytes, so convert to megabytes
>>> mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
>>> wiki = load_dataset("wikimedia/wikipedia", "20220301.en", split="train")
>>> mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

>>> print(f"RAM memory used: {(mem_after - mem_before)} MB")
RAM memory used: 50 MB
```

This is possible because the Arrow data is actually memory-mapped from disk, and not loaded in memory.
Memory-mapping allows access to data on disk, and leverages virtual memory capabilities for fast lookups.

#### Performance

Iterating over a memory-mapped dataset using Arrow is fast. Iterating over Wikipedia on a laptop gives you speeds of 1-3 Gbit/s:

```python
>>> s = """batch_size = 1000
... for batch in wiki.iter(batch_size):
...     ...
... """

>>> elapsed_time = timeit.timeit(stmt=s, number=1, globals=globals())
>>> print(f"Time to iterate over the {wiki.dataset_size >> 30} GB dataset: {elapsed_time:.1f} sec, "
...       f"ie. {float(wiki.dataset_size >> 27)/elapsed_time:.1f} Gb/s")
Time to iterate over the 18 GB dataset: 31.8 sec, ie. 4.8 Gb/s
```


In [12]:
SEED = 42
random.seed(SEED)

## 2. Build a tiny dataset

  
Before we touch any external files or large datasets, we build a tiny in-memory dataset. This is useful as it keeps the workflow:

- Easy to understand (we can see every label example)
- Fast to debug as if something breaks, it's your code and not a data download, file path, or formatting issue.
- Good practice: the steps here (create → inspect → shuffle → sample) are the same steps you'll use on real datasets, just at a smaller scale.

In real projects, most “problems” that can occur are actually data problems (missing fields, inconsistent labels, duplicates, leakage, etc.).


### Our toy task: topic classification

We're going to classify short text snippets into one of four topic labels:

- `sports`
- `tech`
- `politics`
- `finance`

This is a *toy* dataset (only 20 rows), but it's enough to practise the **end-to-end pipeline**:
data → dataset object → inspection → sampling/shuffling → (later) model + evaluation.

### Build the data as a Python list of dictionaries

Each row is a dictionary with:
- **`text`**: the input string (what the model reads)
- **`label`**: the target class (what we want to predict)

Keeping the column names consistent (`text`, `label`) is a small but important habit: later you'll map these into tokenisers and training code that expects stable field names.

In [4]:
records = [
    {"text": "The team won the championship after a close final.", "label": "sports"},
    {"text": "The striker scored two goals in the second half.", "label": "sports"},
    {"text": "The coach announced a new training strategy.", "label": "sports"},
    {"text": "The match was postponed because of heavy rain.", "label": "sports"},
    {"text": "Fans celebrated the upset victory in the playoffs.", "label": "sports"},
    {"text": "The company released a new smartphone with an AI chip.", "label": "tech"},
    {"text": "A security update fixed several critical bugs.", "label": "tech"},
    {"text": "Researchers published a paper on quantum computing.", "label": "tech"},
    {"text": "The startup raised funding to build a robotics lab.", "label": "tech"},
    {"text": "Engineers improved battery life by 30 percent.", "label": "tech"},
    {"text": "The senator introduced a bill on climate policy.", "label": "politics"},
    {"text": "The election debate focused on healthcare reform.", "label": "politics"},
    {"text": "Lawmakers negotiated a budget deal.", "label": "politics"},
    {"text": "A new regulation targets data privacy.", "label": "politics"},
    {"text": "The mayor announced a housing initiative.", "label": "politics"},
    {"text": "Stocks rallied after the interest rate decision.", "label": "finance"},
    {"text": "The company reported higher quarterly earnings.", "label": "finance"},
    {"text": "Analysts warned about inflation risks.", "label": "finance"},
    {"text": "The market dipped after weak retail sales.", "label": "finance"},
    {"text": "Investors are watching the new IPO closely.", "label": "finance"},
]

raw_ds = Dataset.from_list(records)
raw_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 20
})

### What did we just create?



`Dataset.from_list(records)` converts a plain Python list into a Hugging Face `Dataset`, which gives you:

- Column-aware data (you can access `raw_ds["label"]`, etc.)
- Fast operations like `shuffle`, `select`, `map`, `filter`
- A path toward train/validation/test splits later

Even though this feels small, it's the same object type you'll use for datasets with millions of rows from Hugging Face.

### Sanity check: inspect rows




A `Dataset` behaves a bit like a list of dictionaries (row-wise), but it's also column-oriented (like a table).

These are the quickest checks you should *always* do after loading/creating data:

1. Type check: did we actually create a `Dataset`?
2. Length check: how many examples do we have?
3. Row check: what does a single example look like?

In [5]:
type(raw_ds)

datasets.arrow_dataset.Dataset

In [6]:
len(raw_ds) # how many rows

20

In [7]:
raw_ds[0] # first row

{'text': 'The team won the championship after a close final.',
 'label': 'sports'}

In [8]:
raw_ds['text'] # text column

Column(['The team won the championship after a close final.', 'The striker scored two goals in the second half.', 'The coach announced a new training strategy.', 'The match was postponed because of heavy rain.', 'Fans celebrated the upset victory in the playoffs.', ...])

### Quick sampling: shuffle + take a few rows

When datasets get large,we want to look at a few rows to quickly check the samples. To do this we apply the code below where

- `shuffle(seed=SEED)` to randomise order reproducibly
- `select(range(k))` to take the first `k` rows from the shuffled dataset



In [13]:
sample = raw_ds.shuffle(seed=SEED).select(range(3))
for i, row in enumerate(sample):
    print(f"Row {i}:", row)

Row 0: {'text': 'Stocks rallied after the interest rate decision.', 'label': 'finance'}
Row 1: {'text': 'Engineers improved battery life by 30 percent.', 'label': 'tech'}
Row 2: {'text': 'The mayor announced a housing initiative.', 'label': 'politics'}


Alternatively we can always convert to Pandas to quickly look at or explore the data using `to_pandas()`

In [14]:
sample.to_pandas()

,text,label
0,Stocks rallied after the interest rate decision.,finance
1,Engineers improved battery life by 30 percent.,tech
2,The mayor announced a housing initiative.,politics


Before doing any modelling or analysis it can also be useful to check:

- Which labels exist?
- How many examples per label?

With real data, this often reveals problems immediately (missing classes, class imbalance, typos like `"Sport"` vs `"sports"`, etc.).

In [15]:
set(raw_ds["label"]) # returns the set of label values

{'finance', 'politics', 'sports', 'tech'}

In [17]:
raw_ds['label'].count('tech')

5

In [19]:
for lab in set(raw_ds["label"]):
    print(f"{lab}: {raw_ds['label'].count(lab)} examples")

politics: 5 examples
finance: 5 examples
sports: 5 examples
tech: 5 examples


In [18]:
{lab: raw_ds["label"].count(lab) for lab in set(raw_ds["label"])} # returns a dictionary of labels with the count.

{'politics': 5, 'finance': 5, 'sports': 5, 'tech': 5}

## 3. Inspect schema and labels

**Why this matters:**  

A huge number of “model bugs” are actually data + schema bugs.

Common examples:
- You think the input column is called `"text"` but it's actually `"sentence"` or `"review"`.
- Labels are the wrong type (strings vs integers).
- A column contains mixed types (e.g. numbers stored as strings).
- Some rows are missing values, or a column is silently full of `None`.

So before we do any preprocessing or modelling, we inspect the dataset's:
- column names (what fields exist?)
- typed schema (what type does each field claim to be?)
- labels (what are the classes, and are they encoded correctly?)

This is a “measure twice, cut once” step. It's boring, but it prevents hours of confusion later.

In [20]:
raw_ds.column_names

['text', 'label']

### Typed schema: `features`



`features` is the **typed schema** for a `Dataset`.  
It tells you what each column is supposed to be (e.g. `Value("string")`, `ClassLabel`, sequences, audio/image types, etc.).

This is helpful as it improves consistency when you reuse the same pipeline across datasets. It also reduces “silent failures” where something looks fine until training breaks due to wrong type.


In [21]:
raw_ds.features

{'text': Value('string'), 'label': Value('string')}

### Convert string labels to `ClassLabel`

Right now our labels are strings (e.g. `"sports"`).  That's readable for humans, but generally models train on numeric class IDs (0, 1, 2, ...).

Datasets provides `ClassLabel`, which solves two problems at once:

1. Encoding: converts label strings → integers (we will also encounter this again later more broadly)
2. Interpretability: stores the mapping so you can convert integers → strings later

This is important because otherwise it's easy to lose track of what “class 2” actually means when you're debugging predictions.

In [22]:
encoded_ds = raw_ds.class_encode_column("label")
encoded_ds.features


Casting to class labels: 100%|██████████| 20/20 [00:00<00:00, 3540.39 examples/s]


{'text': Value('string'),
 'label': ClassLabel(names=['finance', 'politics', 'sports', 'tech'])}

### Label mapping

Once encoded, the `label` column becomes a `ClassLabel` feature.
We can inspect the available label names and use `int2str` / `str2int` to convert back and forth.

In [23]:
label_feature = encoded_ds.features["label"]
label_feature.names

['finance', 'politics', 'sports', 'tech']

For example we can pull out the integer for 'sports' using `str2int`

In [24]:
label_feature.str2int("sports")

2

Or what label corresponds to the integer 1 with `int2str`.

In [30]:
label_feature.int2str(1)

'politics'

### Example: id ↔ label mapping

We'll take the first row, read its numeric label ID, and convert it back to the original label name.

This is exactly what you'll do later when:
- you evaluate a model and need readable predictions
- you build confusion matrices with class names
- you explain results to a non-technical audience

In [37]:
encoded_ds[0]

{'text': 'The team won the championship after a close final.', 'label': 2}

In [38]:
# Example: id <-> label mapping
example_id = encoded_ds[0]["label"]
example_name = label_feature.int2str(example_id)
example_id, example_name


(2, 'sports')

### Task




1. Pick any two rows (e.g. row 3 and row 17).
2. For each one:
   - read the numeric label (an integer)
   - convert it back to a string with `label_feature.int2str(...)`

Hint: `encoded_ds[i]["label"]` gives the numeric label ID.

In [41]:
i, j = 3, 17
print(i, encoded_ds[i]["label"], label_feature.int2str(encoded_ds[i]["label"]))
print(j, encoded_ds[j]["label"], label_feature.int2str(encoded_ds[j]["label"]))

3 2 sports
17 0 finance


#### Solution

In [42]:
# Example structure (replace i and j with your chosen row indices)
i, j = 3, 17
print(i, encoded_ds[i]["label"], label_feature.int2str(encoded_ds[i]["label"]))
print(j, encoded_ds[j]["label"], label_feature.int2str(encoded_ds[j]["label"]))

3 2 sports
17 0 finance


### 3.5 `Dataset` vs `pandas.DataFrame` - TODO

We will use both of these throughout the module and more broadly the course so it is good to get an understanding of which to use when and what separates them.

- `pandas DataFrame` is best for data analysis / cleaning / joining / summarising.
- `Dataset` is best for ML pipelines it easily allows reproducible preprocessing (`map`), shuffling/splitting, label encoding, and easy hand-off to model training or Hugging Face models.



### They feel similar because…

Both are “tables” i.e. rows (examples) and columns (fields). You can inspect, filter, and transform in both.



### They differ because…
They're optimised for different tasks:
- DataFrame: rich analytics (`groupby`, `merge`, pivots, time series)
- Dataset: ML-safe transforms + typed schema (`features`) + label tools + training integration


#### Example Comparison

Let's convert out current Dataset `raw_ds` to a DataFrame and compare them:

In [43]:
df = raw_ds.to_pandas()

We have seen that to inspect the columns in `raw_ds` we use `.column_names` in DataFrame we use:

In [44]:
df.columns

Index(['text', 'label'], dtype='str')

We can see the similar output as we would expect. Next with Pandas when we want to check the type of data we have we use `dtype`

In [45]:
df.dtypes

text     str
label    str
dtype: object

We can see both are just `object` meaning `string` (usually).

This is much less detailed that the `.feature` schema that allows us to use ML specific types such as `ClassLabel`.

In the following sections we will try to show equivalent ways we can work with a Pandas DataFrame with some of the pros/cons.

## 4. Split data safely with `DatasetDict`




When evaluating a model the evaluation is only meaningful if your train, validation, and test sets are genuinely independent.

If information “leaks” across splits (even subtly), you'll get inflated scores that won't hold up in the real world.

### Leakage intuition (figure)

```text
Bad: preprocess all data together -> split later -> hidden leakage risk
Good: split first -> fit decisions on train -> apply to validation/test
```


`Dataset` has a inbuilt `train_test_split` function in it to allow us to split into training and test set as follows:

In [51]:
encoded_ds.train_test_split(test_size=0.2, seed=SEED)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
})

Above we are specifying that 20% of the data is test set. A random selection is made and this random selection is made reproducible by the `SEED`.

From the above we can see we are returned what looks like a dictionary of Datasets i.e. `DatasetDict`.

In [52]:
split_1 = encoded_ds.train_test_split(test_size=0.2, seed=SEED, stratify_by_column='label')
split_1

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
})

In [53]:
split_1['test']['label']

Column([1, 0, 2, 3])

The above has `stratify_by_column = 'label'` this will result in the training and test datasets having the same proportion of class labels as the original dataset i.e.

If your full dataset is 90% sports and 10% politics, stratify_by_column="label" guarantees that your train set is ~90% sports and your test set is also ~90% sports.

This is very useful.

#### Creating a validation set

To create the validation set (which is usually needed when training our own models to tune hyperparameters) we split the train Dataset.

In [54]:
split_2 = split_1["train"].train_test_split(test_size=0.25, seed=SEED)
split_2

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
})

Note the test size is now 25% as we are splitting the training set which is 80% of the original size. So, 25% of 80% is 20% of the original size.

Combine the training, validation and test set into one `DatasetDict`

In [55]:
ds_dict = DatasetDict({'train': split_2['train'], 'validation': split_2['test'], 'test': split_1['test']})

ds_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
})

## 5. Save and reload datasets




Preprocessing can be expensive in terms of the time it takes and the actually cost involved. Saving intermediate datasets helps by:

- avoiding recomputing the same work repeatedly,
- makes experiments reproducible (everyone trains on the *same* split),
- helps you “freeze” a dataset state for a coursework submission.

Datasets supports `save_to_disk()` which stores the dataset efficiently (Arrow-based) and reloads quickly.

In [56]:
ds_dict.save_to_disk("assets/mini_news_splits")

Saving the dataset (1/1 shards): 100%|██████████| 4/4 [00:00<00:00, 3020.74 examples/s]


If we wanted to reload the data we can also do this easily as follows:

In [57]:
reloaded_splits = DatasetDict.load_from_disk("assets/mini_news_splits")
reloaded_splits

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4
    })
})

In [58]:
# Basic equality check
reloaded_splits["train"][0] == ds_dict["train"][0]


True

## 6. Load local files (CSV/JSONL)

**Why this matters**:
In real projects, our data is often local or stored in a reporsitory but we still want the same pipeline quality as Hub datasets. To deal with this we can use the `load_dataset` function we imported to load the data.


### Quick reference

| Format | Loading script | Example |
| --- | --- | --- |
| CSV | `csv` | `load_dataset("csv", data_files="my_file.csv")` |
| JSON/JSONL | `json` | `load_dataset("json", data_files="my_file.jsonl")` |
| Text | `text` | `load_dataset("text", data_files="my_file.txt")` |


Let's import the data we had in our warm-up exercise last week from here `https://github.com/cm326/Data/raw/refs/heads/master/generated_reviews.csv`

In [59]:
csv_loaded = load_dataset("csv", data_files='/Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/week_3/assets/generated_reviews.csv')

csv_loaded


DatasetDict({
    train: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 80
    })
})

In [62]:
csv_loaded['train'][0]

{'review_id': 1,
 'review_text': 'A couple of Computer Science units were poorly coordinated across different tutors. Some Computer Science modules felt outdated and there was too much overlap between units. There were enough online resources that I rarely needed to buy textbooks.'}

Note: When loading a csv the first row is inferred to be the header i.e. names of columns. If your file does not have names in the first row we can specify them on import as follows:

```python
load_dataset("csv", data_files='path_to_file', column_names=["col1_name", "col2_name",...])
```

By default, one file is loaded as the `train` split. We can pull out this Dataset as before


In [63]:
review_ds = csv_loaded['train']

review_ds

Dataset({
    features: ['review_id', 'review_text'],
    num_rows: 80
})

As before we can look at the `features` schema

In [64]:
review_ds.features

{'review_id': Value('int64'), 'review_text': Value('string')}

We can check a couple of rows

In [65]:
for i, row in enumerate(review_ds.select(range(2))):
    print(f"Row {i}:", row)

Row 0: {'review_id': 1, 'review_text': 'A couple of Computer Science units were poorly coordinated across different tutors. Some Computer Science modules felt outdated and there was too much overlap between units. There were enough online resources that I rarely needed to buy textbooks.'}
Row 1: {'review_id': 2, 'review_text': 'I had to chase staff a few times to get my issue sorted. I had to chase staff a few times to get my issue sorted. Library staff were helpful, especially with referencing and databases.'}


### Task

Split this into a training, validation and test dataset using a 80/10/10 split.

In [66]:
review_1 = review_ds.train_test_split(test_size=0.1, seed=SEED)
review_2 = review_1['train'].train_test_split(test_size=0.1111, seed=SEED)
review_split = DatasetDict({
    'train': review_2['train'],
    'validation': review_2['test'], 'test': review_1['test']})

review_split

DatasetDict({
    train: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 64
    })
    validation: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 8
    })
    test: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 8
    })
})

#### Solution

In [ ]:
review_split1 = review_ds.train_test_split(test_size=0.1, seed=SEED)
review_split2 = review_split1['train'].train_test_split(test_size=0.1111, seed = SEED)

review_split = DatasetDict({
    'train': review_split2['train'],
    'validation': review_split2['test'], 'test': review_split1['test']})

review_split

### Importing Multiple Files

We can give the `load_dataset` a list of file paths or urls and it will import them all with each going into a different `Dataset` in `DatasetDict`.

Note if you already had a separate file for the training, validation and test set then you can let this be known on import using a dictionary and then the keys from this dictionary will be used in `DatasetDict`.

```python
split_files = {
    "train": "path_to_train.csv",
    "validation": "path_to_val.csv",
    "test": "path_to_test.csv",
}

explicit_splits = load_dataset("csv", data_files=split_files)
explicit_splits
```

This would return something like the below.

```python
DatasetDict({
    train: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 64
    })
    validation: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 8
    })
    test: Dataset({
        features: ['review_id', 'review_text'],
        num_rows: 8
    })
})

```

### Load subset of data for experimenting

Sometime we don't want to load all the data at once and in these cases we can load a subset to experiment.

In [67]:
subset = load_dataset("csv", data_files="/Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/week_3/assets/generated_reviews.csv", split="train[:5]")
subset

Dataset({
    features: ['review_id', 'review_text'],
    num_rows: 5
})

### Task

Create your own tiny CSV (at least 10 rows) and load it with `load_dataset`.
Inspect `column_names` and `features`.


### Loading Data from Hugging Face Hub

We can find out info about a dataset we want to use on the Hub too. For example, https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes

This dataset has three splits already of 'train', 'validation' and 'test' we can see this on Hub or via

In [73]:
from datasets import get_dataset_split_names

get_dataset_split_names("cornell-movie-review-data/rotten_tomatoes")

['train', 'validation', 'test']

We can use `load_dataset` to import it all or we can just import a single split like the validation split as follows:

In [74]:
val_split = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="validation")
val_split

Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})

Otherwise we get the whole `DatasetDict` object instead:



In [75]:
movie_ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
movie_ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Check out some of the reviews

In [76]:
for i, row in enumerate(movie_ds['train'].select(range(3))):
    print(f"Review text {i}:", row['text'])

Review text 0: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
Review text 1: the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth .
Review text 2: effective but too-tepid biopic


## 7. Clean and filter with `map` and `filter`




The `map` and `filter` functions allow us to clean, manipulate and filter the data in all of the `Datasets` within a `DatasetDict` easily and in a reproducible way.

For instance, we may want to know if a review is short which we could define to mean less than 10 words. To do this we could add a `feature` to our datasets.

To do this we can create a function that would apply the manipulation to a single example.

In [77]:
def short_review(example):
  text = example['text']
  words = text.split()
  return {'short_review': len(words) < 10}

The function above should classify Review text 2 above as short, let's test it:

In [78]:
movie_ds['train'][2]

{'text': 'effective but too-tepid biopic', 'label': 1}

In [79]:
short_review(movie_ds['train'][2])

{'short_review': True}

We can apply this to our `Datasets` in `DatasetDict` using the `map` function.

In [80]:
movie_ds_sr = movie_ds.map(short_review)
movie_ds_sr

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'short_review'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'short_review'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'short_review'],
        num_rows: 1066
    })
})

Now we could filter our datasets to only contain short reviews using `filter` as follows:

In [81]:
short_reviews_ds = movie_ds_sr.filter(lambda x: x['short_review'])
short_reviews_ds


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'short_review'],
        num_rows: 996
    })
    validation: Dataset({
        features: ['text', 'label', 'short_review'],
        num_rows: 134
    })
    test: Dataset({
        features: ['text', 'label', 'short_review'],
        num_rows: 117
    })
})

In [82]:
for i, row in enumerate(short_reviews_ds['train'].select(range(3))):
  print(f"Review {i}:", row['text'])

Review 0: effective but too-tepid biopic
Review 1: offers that rare combination of entertainment and education .
Review 2: illuminating if overly talky documentary .


### Task

Add a new boolean column `long_review` for examples with more than 50 words.


In [83]:
def long_review(example):
  text = example['text']
  words = text.split()
  return {'long_review': len(words) > 50}

In [85]:
long_review(movie_ds['train'][2])

{'long_review': False}

In [86]:
movie_ds_lr = movie_ds.map(long_review)
movie_ds_lr

Map: 100%|██████████| 1066/1066 [00:00<00:00, 85123.81 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'long_review'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'long_review'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'long_review'],
        num_rows: 1066
    })
})

In [88]:
long_reviews_ds = movie_ds_lr.filter(lambda x: x['long_review'])
long_reviews_ds

Filter: 100%|██████████| 1066/1066 [00:00<00:00, 431409.50 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'long_review'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['text', 'label', 'long_review'],
        num_rows: 2
    })
    test: Dataset({
        features: ['text', 'label', 'long_review'],
        num_rows: 1
    })
})

### Task

Filter out the datasets to only have reviews which are neither long nor short.

In [89]:
def norm_review(example):
  text = example['text']
  words = text.split()
  return {'norm_review': 10 > len(words) > 50}

In [92]:
norm_review(movie_ds['train'][100])

{'norm_review': False}

In [93]:
movie_ds_nr = movie_ds.map(norm_review)
movie_ds_nr

Map: 100%|██████████| 1066/1066 [00:00<00:00, 72928.95 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'norm_review'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'norm_review'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'norm_review'],
        num_rows: 1066
    })
})

In [95]:
norm_reviews_ds = movie_ds_nr.filter(lambda x: x['norm_review'])
norm_reviews_ds

Filter: 100%|██████████| 1066/1066 [00:00<00:00, 457498.01 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'norm_review'],
        num_rows: 0
    })
    validation: Dataset({
        features: ['text', 'label', 'norm_review'],
        num_rows: 0
    })
    test: Dataset({
        features: ['text', 'label', 'norm_review'],
        num_rows: 0
    })
})

Charlie Example Solution

In [96]:
def not_long_short(example):
    return not example['long_review'] and not example['short_review']

In [98]:
movie_ds_sr = movie_ds_sr.map(long_review)
movie_ds_sr

Map: 100%|██████████| 1066/1066 [00:00<00:00, 66972.16 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 1066
    })
})

In [99]:
movie_ds_sr_nlr = movie_ds_sr.filter(not_long_short)
movie_ds_sr_nlr

Filter: 100%|██████████| 1066/1066 [00:00<00:00, 301309.26 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 7519
    })
    validation: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 930
    })
    test: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 948
    })
})

In [100]:
def super_fct(example):
    text = example['text']
    words = text.split()
    if len(words) <= 50:
        return True
    else:
        return False

In [101]:
movie_ds_sr_nlr = movie_ds_sr.filter(super_fct)
movie_ds_sr_nlr

Filter: 100%|██████████| 1066/1066 [00:00<00:00, 283054.45 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 8515
    })
    validation: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 1064
    })
    test: Dataset({
        features: ['text', 'label', 'short_review', 'long_review'],
        num_rows: 1065
    })
})

## 8. Tokenization views in depth (character, word, subword) - TODO This onwards

**Core idea**: models do not process raw text strings directly. They process IDs produced by a tokenizer.

A tokenizer answers one question: *what are the atomic units of text the model should read?*


### Comparison at a glance

| View | What it splits into | Main strengths | Main limitations | Typical use |
| --- | --- | --- | --- | --- |
| Character | single characters | handles typos and rare strings | very long sequences, weak semantic units | specialized or low-resource setups |
| Word | whitespace-delimited words | simple and interpretable | huge vocabulary, many out-of-vocabulary words | older NLP pipelines |
| Subword | learned word pieces | good balance of coverage and sequence length | less interpretable than words | modern transformer models |


### 8.1 Character tokenization

Character tokenization treats every character as a token.

**Why people use it**:
- almost no out-of-vocabulary issue,
- robust to spelling variation,
- simple to implement.

**Main tradeoff**:
- sequences get long quickly, which increases compute and makes long-context modeling harder.


In [ ]:
sample_text = "Tokenizing text is a core task of NLP."

char_tokens = list(sample_text)
print("Character token count:", len(char_tokens))
print("First 25 character tokens:", char_tokens[:25])


### Task

Let's apply character-level tokenisation to our earlier toy example.
1. Standardise text so it is lower case to make things simpler here.
2. Find the vocabulary and it's size. (Only use the 'train' Dataset here.) Remember to add '[UNK]'.
3. For each record tranform the text into a sequence using the token_ids for train, val and test
4. Work out the average length of the tokenised sequence.

#### Solution

In [ ]:
# Character-level vocabulary on our toy corpus
char_vocab = sorted(set(''.join(records_i['text'] for records_i in ds_dict['train']).lower()))
char_vocab.insert(0, '[UNK]')
print("Character vocab size:", len(char_vocab))
print("Vocab entries:", char_vocab)


In [ ]:
# use token_id to create sequences from record text

def char_tokenise(record):
  record['text'] = record['text'].lower()
  token_ids = []
  for char in record['text']:
    if char in char_vocab:
      token_ids.append(char_vocab.index(char))
    else:
      token_ids.append(0)
  return {'tokens': token_ids}

tokenised_ds = ds_dict.map(char_tokenise)
tokenised_ds


In [ ]:
for i, row in enumerate(tokenised_ds['train'].select(range(3))):
  print(f"Review {i}:", row['text'], row['tokens'])

In [ ]:
def tokenised_length(record):
  return {'tokens_len': len(record['tokens'])}

tokenised_ds = tokenised_ds.map(tokenised_length)
tokenised_ds

We can now compute the average length

In [ ]:
# use tokens_len to calculate average length

running_total_len = 0
total_examples = 0

for split_name in tokenised_ds:
  current_dataset = tokenised_ds[split_name]
  running_total_len += sum(current_dataset['tokens_len'])
  total_examples += len(current_dataset)

average_len = running_total_len / total_examples
print(f"Average tokenized length across all splits: {average_len:.2f}")

### 8.2 Word tokenization

Word tokenization usually splits on whitespace.

**Why people use it**:
- easy to explain,
- tokens often map to human-intuitive units,
- shorter sequences than character tokenization.

**Main tradeoffs**:
- punctuation and casing can fragment vocabulary,
- rare words become unknown tokens,
- vocabulary can grow very large.


In [ ]:
word_tokens = sample_text.split()
print("Word token count:", len(word_tokens))
print("Word tokens:", word_tokens)

This has worked okay but when there is lot's of punctuation things are harder as below:

In [ ]:
hard_example = "NLP models, model-driven pipelines, and models!"
print(hard_example.split())

### Task

Let's now apply word-level tokenisation to our earlier toy example.
1. As before standardise text so it is lower case to make things simpler here.
2. Find the vocabulary and it's size. (Only use the 'train' Dataset here.)
3. For each record tranform the text into a sequence using the token_ids for train, val and test
4. Work out the average length of the tokenised sequence.

### 8.3 Subword tokenization

Subword tokenization learns pieces of words from data (for example, prefixes/suffixes and common stems).

**Why modern models use it**:
- far fewer unknown words than pure word tokenization,
- shorter sequences than character tokenization,
- better balance between efficiency and linguistic coverage.

**Main tradeoff**:
- tokens are less human-readable than full words.


### What does "learning" mean in subword tokenization?

In this context, **learning** means learning a **vocabulary of word pieces** and rules for how to split text into those pieces.

Important: this is **not** the transformer learning task itself.
It is a separate preprocessing stage that is usually done before model pretraining or fine-tuning.


###   How is subword learning done?

Subword vocabularies are typically learned from a large text corpus using frequency/statistical objectives.
Common algorithms include:

- **BPE (Byte Pair Encoding)**: repeatedly merge frequent symbol pairs.
- **WordPiece**: choose merges that best improve likelihood of training text.
- **Unigram (SentencePiece)**: start with many candidates, then remove low-utility pieces.

The result is a tokenizer that can represent frequent words efficiently while still handling unseen words via smaller pieces.


### Tokenizer learning vs Transformer learning

| Aspect | Tokenizer learning | Transformer learning |
| --- | --- | --- |
| What is learned | token vocabulary + split behavior | neural weights (embeddings, attention, MLP, heads) |
| Typical objective | compression / likelihood over text pieces | predict labels/tokens via gradient descent |
| Output artifact | tokenizer files + token-id mapping | model weight files + config |
| Frequency | usually trained once per tokenizer | trained/fine-tuned many times across tasks |

A useful mental model:
- tokenizer decides the **alphabet of pieces**,
- transformer learns how those pieces combine into meaning and predictions.


Let's now try fitting a BPE tokeniser of our own on our toy data again using the 'train' data to do this.

Luckily, there is a `tokenizers` package from Hugging Face which will help us do this.

**Note:** In relality we won't often need to fit our own tokenisation method as we have to use the ones from the pretrained model we want to use. We shall see this too later.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(vocab_size=500, special_tokens=["[UNK]"])

def batch_iterator():
    for example in ds_dict['train']:
        yield example["text"]

tokenizer.train_from_iterator(batch_iterator(), trainer)

The learnt vocab size is given below:

In [ ]:
tokenizer.get_vocab_size()

We can also see the vocab:

In [ ]:
vocab = tokenizer.get_vocab()
vocab

Apply this to our `ds_dict`.

In [ ]:
subword_token_dict = ds_dict.map(lambda x: {"tokens": tokenizer.encode(x["text"]).ids})
subword_token_dict

Let's check out a few rows:

In [ ]:
for i, row in enumerate(subword_token_dict['train'].select(range(3))):
  print(f"Review {i}:", row['text'], row['tokens'])

### Task

Work out the average tokenised length now.

### Task



Let's extend things to a larger dataset. Go through the comparison we have above of character, word and sub-word on the moview review data. Again use the training data for this.

<!-- Your answer here -->


## 9. Subword tokenization with a pretrained Hugging Face tokenizer

We now use a real tokenizer from a pretrained checkpoint.

**Why this matters**:
When you fine-tune a pretrained model, you should almost always use the tokenizer from the same checkpoint.


### Does every Hugging Face model have its own tokenizer?

Usually, each **checkpoint** on the Hub provides a compatible tokenizer.
However, not every checkpoint has a unique tokenizer:

- many fine-tuned checkpoints reuse the base model tokenizer,
- several checkpoints in one model family may share tokenizer files,

Practical rule: load tokenizer and model from the same checkpoint family unless you have a strong reason not to.


### Anatomy of a Hugging Face text model package




For NLP, think in two main parts plus metadata:

1. **Tokenizer artifacts**: vocabulary + split rules + special-token mapping.
2. **Model artifacts**: config + learned weights.

Typical runtime flow:

```text
Raw text -> Tokenizer -> input_ids -> Model
```

In Hugging Face code, this often means:
- `AutoTokenizer.from_pretrained(checkpoint)`
- `AutoModel` or task-specific classes like `AutoModelForSequenceClassification`

For multimodal models, Hugging Face may use `AutoProcessor` (tokenizer + image/audio preprocessing together).

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)



In [ ]:
sample_text = "Tokenizing text is a core task of NLP."
subword_tokens = tokenizer.tokenize(sample_text)
print("Subword token count:", len(subword_tokens))
print("Subword tokens:", subword_tokens)


In [ ]:
encoded = tokenizer(sample_text)
ids = encoded["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(ids)
list(zip(tokens, ids))


### Pros and cons recap: subword tokenization

**Pros**:
- strong default choice for pretrained transformers,
- handles rare and unseen words better than word-level tokenization,
- generally efficient sequence lengths.

**Cons**:
- less intuitive than full words,
- sequence length can still grow for noisy text,
- tokenizer/model mismatch causes quality drops.


### Attention mask intuition




An attention mask tells the model:

“Which tokens are real text, and which are just padding?”

```text
input_ids:      [101,  2023, 2003, 1037, 3231, 102,    0,    0]
attention_mask: [  1,     1,    1,    1,    1,   1,    0,    0]

1 = real token, 0 = padding token
```

Neural networks require inputs in a fixed-size batch tensor.

But sentences have different lengths:

```text
"I love it"
"This was the worst film I have ever seen"
```

To batch them together, we pad the shorter one:

```text
[I, love, it, PAD, PAD, PAD]
[This, was, the, worst, film, I]
```

Now they're same length. But we dont want the model to treat PAD as meaningful So we create the attention mask:

```text
attention_mask:
[1, 1, 1, 0, 0, 0]
[1, 1, 1, 1, 1, 1]
```

Inside the Transformer:

- Self-attention computes scores between tokens.

Without masking:

- The model could attend to padding tokens.

That introduces noise.

- It distorts attention distributions.

With masking:

- Padding positions are given a large negative value (like -inf) before softmax activation.

After softmax, they effectively get weight 0 so the model 'ignores' them.

In [ ]:
pair = tokenizer(
    ["short text", "this is a longer text example"],
    padding=True,
    truncation=True,
)
pair


In the above we can see that:

For the first sentence:

[1, 1, 1, 1, 0, 0, 0, 0]

Meaning: First 4 tokens are real and Last 4 are padding

For the second sentence:

[1, 1, 1, 1, 1, 1, 1, 1]

No padding.

The `token_type_ids` are used essentially to identify tokens from difference sentences. In this case the inputs are only one sentence long. Many modern models often ignore this entirely anyway.

### Task

Tokenize two custom sentences with the pretrained tokenizer.
1. Compare token counts.
2. Identify one word that was split into multiple subword pieces.


## 10. Tokenize the full dataset

Now we apply tokenization to the whole dataset so it becomes model-ready. Note below padding is False. If we wanted to run the data through a model in batches (we usually do) then we should be using padding here.


In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding=False,
        truncation=True,
        max_length=128,
    )

tokenized = ds_dict.map(tokenize_batch, batched=True)
tokenized["train"][0]


In [ ]:
tokenized.column_names


In [ ]:
def add_input_length(batch):
    return {"input_length": [len(ids) for ids in batch["input_ids"]]}

tokenized = tokenized.map(add_input_length, batched=True)
tokenized["train"][0]


### Figure: token length distribution

Length checks help pick sensible truncation settings.


In [ ]:
import matplotlib.pyplot as plt

lengths = tokenized["train"]["input_length"]
plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=8)
plt.title("Tokenized Length Distribution (Train)")
plt.xlabel("input_length")
plt.ylabel("count")
plt.show()



### Task

Use the tokeniser to tokenise the `movie_ds` from earlier.

### Prepare schema for training APIs

Most Hugging Face training utilities expect:
- `input_ids`
- `attention_mask`
- `labels`


In [ ]:
model_ready = tokenized.remove_columns(["text"])
model_ready = model_ready.rename_column("label", "labels")
model_ready


### Task

Compute average `input_length` by class label on the train split.


## 11. Save processed dataset

Save model-ready data so later experiments do not repeat preprocessing.


In [ ]:
path =  "assets/mini_news_tokenized"

model_ready.save_to_disk(path)

reloaded_processed = DatasetDict.load_from_disk(path)
reloaded_processed


## 12. Further exercises

Use these to confirm you can run an end-to-end workflow independently.


### Exercise 1: Custom split policy

Create 70/15/15 splits with a new seed and print class distributions for each split for our original data


### Exercise 2: Model-switch tokenization

Try a different checkpoint (for example `distilbert/distilroberta-base`).
Compare average token length vs `distilbert-base-uncased`.


### Exercise 3: Public dataset warm-up

Load `stanfordnlp/imdb` with `load_dataset`.
Print split sizes, column names, and one sample.


### Exercise 4: Bring your own data to Hub format

Take a small dataset you create, convert it to `DatasetDict`, tokenize it, and save it to disk.
If possible, prepare a dataset card draft.


## Key takeaways

- `Dataset` and `DatasetDict` provide structure, speed, and reproducibility.
- `map` and `filter` are the backbone of NLP preprocessing pipelines.
- Tokenization bridges raw text and model inputs.
- Putting your own data in `datasets` format makes model experimentation much easier.


## References

- MIT 15.773 Lecture 6: Embeddings  
  https://ocw.mit.edu/courses/15-773-hands-on-deep-learning-spring-2024/mit15_773_s24_lec06.pdf
- Hugging Face Datasets: Load guide  
  https://huggingface.co/docs/datasets/loading
- Hugging Face LLM Course: Datasets chapter  
  https://huggingface.co/learn/llm-course/chapter5/1
- Deep Learning with Python: Chapter 14 Text Classification (F.Chollet & M. Watson) https://deeplearningwithpython.io/chapters/chapter14_text-classification/
- Natural Language Processing with Transformers: Chapter 2 (https://github.com/nlp-with-transformers/notebooks/blob/main/02_classification.ipynb)
